# M3 video (V1): synthetic clips, per-frame verdicts and smoothing

We turn 200 validation photos into damaged synthetic clips and score the verdict the screen would show. This is V1:
how often does the verdict flip on video, how soon is it stable, and which way of turning frames into a screen
verdict (per frame, an EMA, or 4-of-6 voting) works best?

**Clips.** Each photo (classes in turn, seeded) becomes a 10-second, 15 FPS clip by M1 section 3.4's recipe: camera
path (pan, zoom, rotation, hand shake), motion blur, low light and noise, JPEG, and leaves hiding 30% to 70% of the
snake. The scene is the raw photo; the live step fixes each frame's exposure. The leaves go over the front end's
predicted mask from the notebook 13 kernel, never a SAM mask; a photo without a usable predicted mask gets them at
seeded spots in a fixed central ellipse.

**Scoring.** One trained run's model reads the undistorted photo (the clean control) and every frame on the GPU, and
each verdict comes from `decision.decide` at the thresholds `decision_report` fits on that run's own calibration
outputs (`fitted_decision`, with the same `RECALL_BOUND` as the run's decision file). Three methods turn frames into
what the screen shows: the per-frame verdict, an EMA of the probabilities (alpha 0.3) and 4-of-6 voting on the
verdict labels, which shows "unknown" whenever no label fills 4 of the last 6 frames. Each is scored per clip, for the
verdict and for the species: flips, time to the first stable label and to its confirmation, whether the last stable
label is right, how often a dangerous snake's clip ever shows "not listed", and the same clip's clean-photo verdict
beside it.

**Box-crop runs.** A box_crop (F1) run reads every frame, and the clean photo, through notebook 13's D1 detector
first, as the pipeline does: the frame's best box is cropped with the same calls `pipeline` makes, and a frame whose
box is under the decision file's `tau_snake` (or that has none) is a "no snake found" frame, smoothed as `pipeline`
smooths one. The detector is `frontends/D1/<winner>/detector.pt` in `PREDS_KERNEL`'s output, named by notebook 13's
`winners.json`; `tau_snake` comes from fitting the decision file with the front end's box confidences.
`video_metrics.json` counts the no-snake frames per clip and overall.

**Schedules.** Two schedules are scored from the same frame outputs. `every_frame` reads all 150 frames. `laptop_cpu`
plays the clip live at the per-frame times of the slowest timing run in `data/m3/latency/<LATENCY_BACKEND>/`, measured
on the laptop CPU we use as the O6 device, so the model skips the frames that arrive while it is busy, as it would
there. A box_crop run's frames also go through the detector, so its `laptop_cpu` plays the deployed pipeline's
one-thread time from X1's `results/m3/x1_laptop/package.csv` (the deployment row for its encoder with YOLO26n at
320 px); `run_info.laptop_latency` names the row and figure used.

**Inputs:** the dataset `serpentguard-indian-snakes-clean`, found with `kagglehub.dataset_download`. The run and the
predicted masks come from the outputs of the kernels `RUN_KERNEL` and `PREDS_KERNEL`, through
`kagglehub.notebook_output_download`: a batch kernel can read another notebook's output only when that notebook is
declared as a kernel source. `RUN_ID` names the run folder (`best.pt`, `config.yaml`, `subset_hash.txt`,
`prototypes.npz` and the calibration, validation and unknown outputs). The code comes from `../src` and `../data`,
which the runner kernel unpacks into `/kaggle/working/repo`.

**Outputs** in `/kaggle/working/video/`: `video_metrics.json` (recipe, thresholds, versions, run info, every clip's
generation parameters and the per-schedule, per-method summary), `video_clips.csv` (one row per clip, schedule and
method, added as each clip finishes), `video_frames.npz` (per clip and schedule: the processed frames, when each
shows, and every method's verdict and top-1, so metrics can be recomputed without the model) and `summary.csv`. No
frame or clip is saved.

**Run:** on Kaggle with one GPU. `N_CLIPS` and `SEED` set the clips, `LATENCY_BACKEND` the laptop timings and
`RECALL_BOUND` the decision file's fit; `LIMIT` makes a smoke run. A photo that fails is counted and skipped.


In [1]:
RUN_KERNEL = 'dataset:puneetbajajiitm/serpentguard-m3-run-final'
RUN_ID = 'H1_top1_s1'
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
N_CLIPS = 200
SEED = 42
LATENCY_BACKEND = "torch_fp32_cpu"  # data/m3/latency/<backend>/: laptop timing runs, the slowest is played for a
                                    # full-frame run; a box_crop run plays X1's deployed pipeline
LIMIT = None                        # first N clips, for a smoke run
RECALL_BOUND = 'lower'

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the eval normalisation runs through it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
ULTRALYTICS_VERSION = "8.4.202"    # requirements-m3.txt; the D1 detector a box_crop run reads every frame through
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm
import ultralytics

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}, ultralytics {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
albumentations 2.0.8, timm 1.0.31, ultralytics 8.4.202


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; the decision report's metrics use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import shutil
import time
from dataclasses import asdict
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import numpy as np
import pandas as pd
import torch

from serpentguard.config import load_config
from serpentguard.data import ID_TYPES, build_frame, with_files
from serpentguard.decision import DECISION_FILE
from serpentguard.frontend import D1, WINNERS_FILE
from serpentguard.run_data import BOX_CROP, FRONTEND_PREDS_FILE, input_mode
from serpentguard.train import (CALIBRATION_OUTPUTS, CHECKPOINT_FILE, CONFIG_FILE, HASH_FILE, PROTOTYPES_FILE,
                                UNKNOWN_OUTPUTS, VAL_OUTPUTS)
from serpentguard.video import round_robin

## The V1 harness

The clip recipe, the frame schedule and the 4-of-6 vote come from `serpentguard.video`, which the app and notebook 15
share. The rest of V1 lives here: the fitted decision, the EMA and the three methods; reading each frame through
the run's model and the laptop schedule; and the run itself, with its report and files.

In [5]:
import json
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import serpentguard
from serpentguard.data import apply_mode, fix_exposure, load_mask, prepare
from serpentguard.decision import (HEAD, NOT_LISTED, PLAIN, POINT, DangerTables, Thresholds, danger_tables, decide,
                                   decision_report, expected_verdict)
from serpentguard.frontend import load_detector, predict_boxes
from serpentguard.medical_importance import MEDICAL_CSV
from serpentguard.metrics import UNKNOWN, VERDICTS, assert_no_test_paths, grouped_bootstrap
from serpentguard.models import INPUT_SIZE
from serpentguard.run_data import BOX_CROP, input_mode
from serpentguard.run_files import PROTOTYPES_FILE
from serpentguard.run_outputs import (PREDICTION_KEYS, file_sha256, library_versions, load_checkpoint,
                                      photographer_groups, softmax, subset_hash, write_json)
from serpentguard.video import (CENTRE_REGION, CLIP_FPS, CLIP_MAX_SIDE, CLIP_SECONDS, EXPOSURE_RANGE,
                                JPEG_QUALITY_RANGE, LEAF_ASPECT_RANGE, LEAF_COVER_RANGE, LEAF_GREEN_HIGH,
                                LEAF_GREEN_LOW, LEAF_LENGTH_RANGE, LEAF_POINTS, LEAF_TRIES, LIGHT_GAIN_RANGE,
                                MAX_BLUR_PX, MIDRIB_DARKEN, NOISE_SIGMA_RANGE, PAN_MAX, PATH_KEYPOINTS,
                                ROTATION_MAX_DEG, SEED, SHAKE_PERSISTENCE, SHAKE_RANGE, SHAKE_ROTATION_DEG, SRGB_GAMMA,
                                STABLE_SECONDS, TARGET_FPS, VOTE, VOTE_K, VOTE_N, ZOOM_RANGE, load_raw_rgb,
                                normalised, schedule, synthesize_clip, video_metrics, vote_k_of_n)

EMA_ALPHA = 0.3
NO_SPECIES = -1                   # the species a vote shows while no species has k of the last n frames
O6_MAX_FLIPS, O6_STABLE_WITHIN = 1, 2.0
PROB_FLOOR = 1e-12
FULL_INPUT = "full"
MS_PER_S = 1000.0
PER_FRAME, EMA = "per_frame", "ema"
METHODS = (PER_FRAME, EMA, VOTE)
VERDICT_CODE = {label: code for code, label in enumerate(VERDICTS)}
VIDEO_FILE, CLIPS_FILE, FRAMES_FILE = "video_metrics.json", "video_clips.csv", "video_frames.npz"
NO_FRONT_END = "none: V1 has no detector, so every frame counts as showing a snake"
FRAME_MODES = (FULL_INPUT, BOX_CROP)
NO_SNAKE_RULE = ("a frame whose best D1 box is under the decision file's tau_snake, or that has none, has no snake: "
                 "SnakeID does not run, the per-frame and EMA methods show 'unknown' and no species there, voting "
                 "counts it as 'unknown', and the EMA runs over the frames that found a snake, as pipeline._run_clip "
                 "does (R111)")
REPO_ROOT = Path(serpentguard.__file__).resolve().parents[2]
LATENCY_DIR = REPO_ROOT / "data" / "m3" / "latency"  # <backend>/<UTC time>.json: one measured timing run per file
PACKAGE_CSV = REPO_ROOT / "results" / "m3" / "x1_laptop" / "package.csv"  # X1's laptop package table
DEPLOYMENT, DEPLOYED_FRONT_END, DEPLOYED_FIGURE = "deployment", "yolo26n-ft@320", "pipeline_ms_1t"
DEPLOYED_KEYS = ("precision", "encoder", "snakeid_int8", "size", "front_end", "front_end_precision", "pipeline_ms_1t",
                 "pipeline_ms_4t", "latency_device")

RECIPE = {
    "source": "M1 section 3.4 gives seconds, fps and leaf_cover; the rest are chosen in serpentguard.video",
    "seconds": CLIP_SECONDS, "fps": CLIP_FPS, "leaf_cover": LEAF_COVER_RANGE, "max_side": CLIP_MAX_SIDE,
    "scene": "the raw photo, without the exposure fix; the live step fixes each frame",
    "path_keypoints": PATH_KEYPOINTS, "easing": "cosine between keypoints: the camera slows into each pose",
    "pan_max": PAN_MAX, "zoom": ZOOM_RANGE, "rotation_max_deg": ROTATION_MAX_DEG,
    "border": "reflect_101: where pan or rotation reaches past the photo's edge, even at zoom 1, the photo is "
              "mirrored into view",
    "shake": SHAKE_RANGE, "shake_rotation_deg": SHAKE_ROTATION_DEG, "shake_persistence": SHAKE_PERSISTENCE,
    "shake_model": "AR(1) on x, y and rotation, unit variance, scaled by the clip's shake",
    "exposure": EXPOSURE_RANGE, "max_blur_px": MAX_BLUR_PX,
    "blur": "a line kernel along the frame centre's image motion since the last frame, times the exposure",
    "light_gain": LIGHT_GAIN_RANGE, "srgb_gamma": SRGB_GAMMA, "noise_sigma": NOISE_SIGMA_RANGE,
    "noise": "gaussian per pixel and channel, sigma / sqrt(light gain)", "jpeg_quality": JPEG_QUALITY_RANGE,
    "leaf_length": LEAF_LENGTH_RANGE, "leaf_aspect": LEAF_ASPECT_RANGE, "leaf_points": LEAF_POINTS,
    "leaf_tries": LEAF_TRIES, "leaf_green_low": LEAF_GREEN_LOW, "leaf_green_high": LEAF_GREEN_HIGH,
    "midrib_darken": MIDRIB_DARKEN,
    "occluder": "the front end's predicted mask, never SAM (R79); without a usable one, seeded spots in a central "
                "ellipse",
    "centre_region": CENTRE_REGION,
    "order": "leaves on the scene, camera warp, motion blur, low light, noise, JPEG",
}
METHOD_TEXT = {PER_FRAME: "each frame's own verdict and top-1",
               EMA: f"EMA (alpha {EMA_ALPHA}) of every head's probabilities and the embedding, then decide",
               VOTE: f"a label shows while it fills {VOTE_K} of the last {VOTE_N} per-frame labels; otherwise "
                     f"'{UNKNOWN}' (species {NO_SPECIES}), never a stale label (R87)"}


@dataclass(frozen=True)
class FittedDecision:
    """decision_report's thresholds with the prototypes, tables, scorer and recall bound they were fitted with."""
    thresholds: Thresholds
    prototypes: np.ndarray
    tables: DangerTables
    prototype_set: str = PLAIN
    scorer: str = HEAD
    prototype_scale: float | None = None
    recall_bound: str = POINT

    def decided(self, outputs):
        return decide(outputs, self.thresholds, self.prototypes, self.tables, scorer=self.scorer,
                      scale=self.prototype_scale)


def fitted_decision(run_dir, classes, metadata=None, prototype_set=PLAIN, medical=None, scorer=HEAD,
                    recall_bound=POINT, box_conf=None):
    """decision_report on the run (it writes decision_metrics.json into run_dir), and what it fitted."""
    run_dir = Path(run_dir)
    medical = pd.read_csv(MEDICAL_CSV, keep_default_na=False) if medical is None else medical
    report = decision_report(run_dir, classes, metadata, medical=medical, prototype_set=prototype_set, scorer=scorer,
                             recall_bound=recall_bound, box_conf=box_conf)
    with np.load(run_dir / PROTOTYPES_FILE) as prototypes:
        chosen = prototypes[prototype_set]
    return FittedDecision(Thresholds(**report["thresholds"]), chosen, danger_tables(classes, medical), prototype_set,
                          report["scorer"], report.get("prototype_scale"), report["recall_bound"])


def smooth_ema(probs_seq, alpha=EMA_ALPHA):
    """s_0 = p_0 and s_t = alpha p_t + (1 - alpha) s_(t-1), along the first axis."""
    values = np.asarray(probs_seq, dtype=float)
    smoothed = values.copy()
    for index in range(1, len(values)):
        smoothed[index] = alpha * values[index] + (1 - alpha) * smoothed[index - 1]
    return smoothed


def smooth_outputs(outputs, thresholds, alpha=EMA_ALPHA):
    """The EMA of the embedding and of each head's probabilities, as logits: T log(p) gives p back in decide."""
    temperatures = {"species_logits": thresholds.temperature, "who_logits": thresholds.temperature_who,
                    "genus_logits": 1.0, "family_logits": 1.0}
    smoothed = {"embedding": smooth_ema(outputs["embedding"], alpha)}
    for name, temperature in temperatures.items():
        if name in outputs:
            probs = smooth_ema(softmax(np.asarray(outputs[name], dtype=float) / temperature), alpha)
            smoothed[name] = temperature * np.log(np.maximum(probs, PROB_FLOOR))
    return smoothed


def clip_methods(outputs, decision, alpha=EMA_ALPHA, k=VOTE_K, n=VOTE_N, snake=None):
    """Each method's verdicts and species ids per frame; outputs hold only the `snake` frames (NO_SNAKE_RULE)."""
    found = np.ones(len(outputs["species_logits"]), dtype=bool) if snake is None else np.asarray(snake, dtype=bool)
    rows = np.flatnonzero(found)
    labels = {method: [UNKNOWN] * len(found) for method in (PER_FRAME, EMA)}
    top1 = {method: [NO_SPECIES] * len(found) for method in (PER_FRAME, EMA)}
    if len(rows):
        decided = {PER_FRAME: decision.decided(outputs),
                   EMA: decision.decided(smooth_outputs(outputs, decision.thresholds, alpha))}
        for method, result in decided.items():
            for position, row in enumerate(rows):
                labels[method][row], top1[method][row] = str(result.label[position]), int(result.top1[position])
    verdicts = {**labels, VOTE: vote_k_of_n(labels[PER_FRAME], k, n, fallback=UNKNOWN)}
    species = {**top1, VOTE: vote_k_of_n(top1[PER_FRAME], k, n, fallback=NO_SPECIES)}
    return verdicts, species

In [6]:
def frame_input(rgb, size=INPUT_SIZE):
    """One frame as the model reads it: exposure fix, letterbox, ImageNet normalisation; (1, 3, size, size)."""
    return normalised(prepare(fix_exposure(rgb), None, FULL_INPUT, size)[0])


def torch_predictor(run_dir, device="cpu", detector=None, tau_snake=None):
    """predict_frame for a run's best.pt; a box-crop run reads each frame through the D1 detector as the pipeline does
    and returns None for a frame with no snake."""
    cfg, model, _ = load_checkpoint(run_dir, device)
    mode = input_mode(cfg)
    if mode not in FRAME_MODES:
        raise ValueError(f"{cfg.run_id} reads {mode} input, which needs a segmenter on every frame")
    described = {"detector": type(detector).__name__}
    if isinstance(detector, (str, Path)):
        weights = Path(detector)
        described = {"detector": str(weights), "detector_sha256": file_sha256(weights)}
        detector = load_detector(weights.parent, {"weights": weights.name})

    @torch.inference_mode()
    def predict_frame(rgb):
        if mode == FULL_INPUT:
            tensor = frame_input(rgb, cfg.input_size)
        else:
            fixed = fix_exposure(rgb)
            found = predict_boxes(detector, [fixed])[0]
            if found is None or found[1] < tau_snake:
                return None
            tensor = normalised(apply_mode(fixed, None, mode, found[0], cfg.input_size)[0])
        outputs = model(tensor.to(device))
        return {name: outputs[name].float().cpu().numpy() for name in PREDICTION_KEYS if outputs.get(name) is not None}

    if detector is not None:
        predict_frame.front_end = {"mode": mode, **described, "tau_snake": tau_snake, "no_snake": NO_SNAKE_RULE}
    return predict_frame


def laptop_schedule(cfg, backend, package_csv=PACKAGE_CSV, root=LATENCY_DIR):
    """(seconds per frame, source): the slowest timing run for a full-frame run, X1's deployed pipeline for F1."""
    if input_mode(cfg) != BOX_CROP:
        runs = [(path, json.loads(path.read_text())) for path in sorted((Path(root) / backend).glob("*.json"))]
        path, run = max(runs, key=lambda found: found[1]["median_ms"])
        return np.asarray(run["seconds"]), {"file": _repo_path(path), **{k: v for k, v in run.items() if k != "seconds"}}
    table = pd.read_csv(package_csv)
    rows = table[(table["precision"] == DEPLOYMENT) & (table["encoder"] == cfg.encoder)
                 & (table["front_end"] == DEPLOYED_FRONT_END)]
    assert len(rows) == 1, f"{package_csv} must hold one {DEPLOYMENT} row for {cfg.encoder}; found {len(rows)}"
    row = rows.iloc[0]
    record = {"file": _repo_path(package_csv), "figure": DEPLOYED_FIGURE,
              **{key: row[key].item() if hasattr(row[key], "item") else row[key] for key in DEPLOYED_KEYS}}
    return float(row[DEPLOYED_FIGURE]) / MS_PER_S, record


def _repo_path(path):
    path = Path(path).resolve()
    return str(path.relative_to(REPO_ROOT)) if path.is_relative_to(REPO_ROOT) else str(path)

In [7]:
def run_video(photos, predict_frame, decision, out_dir, schedules, seed=SEED, run_info=None, checkpoint=None):
    """One seeded clip per photo, scored per schedule and method; writes the three V1 files and returns the report."""
    assert_no_test_paths(photos)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / CLIPS_FILE).unlink(missing_ok=True)
    front_end = getattr(predict_frame, "front_end", None)
    records, errors = {}, []
    for index, photo in enumerate(photos.itertuples(index=False)):
        try:
            records[photo.path], rows = _clip(photo, seed + index, predict_frame, decision, schedules)
        except Exception as error:  # one bad photo must not end an unattended run; it is counted in the report
            errors.append({"path": photo.path, "error": f"{type(error).__name__}: {error}"})
            print(f"clip {index} ({photo.path}) failed: {errors[-1]['error']}", flush=True)
            continue
        pd.DataFrame(rows).to_csv(out_dir / CLIPS_FILE, mode="a", header=not (out_dir / CLIPS_FILE).exists(),
                                  index=False)
    if not records:
        raise RuntimeError(f"no clip finished: {len(errors)} photos failed, the first {errors[:1]}")
    _save_records(out_dir / FRAMES_FILE, records)
    clips = pd.read_csv(out_dir / CLIPS_FILE, dtype={"observer_id": str})
    done = [records[path] for path in photos["path"] if path in records]
    params = [record["params"] for record in done]
    finished = photos[photos["path"].isin(records)]
    report = {"run_info": run_info or {}, "photos": len(photos), "clips": len(records), "errors": errors,
              "classes": int(finished["class_id"].nunique()), "seed": seed, "photos_hash": subset_hash(photos["path"]),
              "checkpoint_sha256": None if checkpoint is None else file_sha256(checkpoint),
              "recipe": RECIPE, "methods": METHOD_TEXT,
              "stability": f"a label is stable once it stays {STABLE_SECONDS} s; time_to_stable is when the first "
                           f"stable label shows, time_to_confirmed when it has stayed {STABLE_SECONDS} s, "
                           f"clip_correct whether the last stable label is right",
              "decision": {"thresholds": asdict(decision.thresholds), "prototype_set": decision.prototype_set,
                           "scorer": decision.scorer, "prototype_scale": decision.prototype_scale,
                           "recall_bound": decision.recall_bound,
                           "front_end": front_end or NO_FRONT_END,
                           "expected_verdict": "dangerous for a danger-table species, not_listed otherwise; unknown "
                                               "is never right"},
              "no_snake": _no_snake_summary(done),
              "clean_control": "clean_* columns: the model on the undistorted photo, through the same live step",
              "o6_targets": {"fps": TARGET_FPS, "max_flips": O6_MAX_FLIPS, "stable_within_seconds": O6_STABLE_WITHIN},
              "versions": library_versions(),
              "occluder_regions": dict(Counter(clip["occluder_region"] for clip in params)),
              "leaf_cover_below_range": sum(clip["leaf_cover"] < LEAF_COVER_RANGE[0] for clip in params),
              "schedules": {name: _schedule_summary(clips[clips["schedule"] == name], latency)
                            for name, latency in schedules.items()},
              "clip_params": params}
    return write_json(out_dir / VIDEO_FILE, report)


def _clip(photo, clip_seed, predict_frame, decision, schedules):
    raw = load_raw_rgb(photo.image_file)
    clean_outputs = predict_frame(raw)
    clean = None if clean_outputs is None else decision.decided(clean_outputs)
    clip = synthesize_clip(raw, load_mask(getattr(photo, "mask_file", "") or "", raw.shape[:2]), seed=clip_seed)
    found = [predict_frame(frame) for frame in clip.frames]
    snake = np.array([outputs is not None for outputs in found])
    with_snake = [outputs for outputs in found if outputs is not None]
    outputs = {name: np.concatenate([np.atleast_2d(out[name]) for out in with_snake])
               for name in (with_snake[0] if with_snake else ())}
    row_of = np.cumsum(snake) - 1  # each snake frame's row in outputs
    record = {"params": clip.params, "clean_verdict": UNKNOWN if clean is None else str(clean.label[0]),
              "clean_top1": NO_SPECIES if clean is None else int(clean.top1[0]),
              "no_snake_frames": int((~snake).sum()), "schedules": {}}
    rows = []
    for name, latency in schedules.items():
        processed, shown = schedule(len(clip.frames), CLIP_FPS, latency)
        kept = snake[processed]
        chosen = {key: value[row_of[processed[kept]]] for key, value in outputs.items()}
        verdicts, species = clip_methods(chosen, decision, snake=kept)
        record["schedules"][name] = {"processed": processed, "shown": shown, "snake": kept, "verdicts": verdicts,
                                     "species": species}
        rows += _clip_rows(photo, record, name, decision.tables)
    return record, rows


def _clip_rows(photo, record, schedule_name, tables):
    params, frames = record["params"], record["schedules"][schedule_name]
    class_id, end = int(photo.class_id), params["frames"] / params["fps"]
    dangerous, expected = bool(tables.dangerous[class_id]), expected_verdict(class_id, tables)
    observer = getattr(photo, "observer_id", None)
    rows = []
    for method in METHODS:
        verdicts = frames["verdicts"][method]
        scores = {"verdict": video_metrics(verdicts, expected, frames["shown"], end),
                  "species": video_metrics(frames["species"][method], class_id, frames["shown"], end)}
        rows.append({"path": photo.path, "class_id": class_id, "observer_id": "" if pd.isna(observer) else str(observer),
                     "clip_seed": params["seed"], "schedule": schedule_name, "method": method,
                     "frames": params["frames"], "frames_processed": len(frames["processed"]),
                     "no_snake_frames": record["no_snake_frames"],
                     "no_snake_processed": int((~np.asarray(frames["snake"], dtype=bool)).sum()),
                     "effective_fps": len(frames["processed"]) / end, "dangerous": dangerous,
                     "expected_verdict": expected, "ever_not_listed": NOT_LISTED in verdicts,
                     "clean_verdict": record["clean_verdict"], "clean_top1": record["clean_top1"],
                     "clean_verdict_correct": record["clean_verdict"] == expected,
                     "clean_species_correct": record["clean_top1"] == class_id,
                     **{f"{kind}_{key}": value for kind, result in scores.items() for key, value in result.items()}})
    return rows


def _save_records(file, records):
    """FRAMES_FILE: every clip's per-frame arrays end to end, cut at `offsets`, enough to recompute any clip metric."""
    clips = list(records)
    entries = [(path, name, frames) for path in clips for name, frames in records[path]["schedules"].items()]
    lengths = [len(frames["processed"]) for _, _, frames in entries]
    arrays = {"verdict_names": np.array(VERDICTS), "clip_paths": np.array(clips),
              "clip_params": np.array([json.dumps(records[path]["params"]) for path in clips]),
              "clean_verdict": np.array([VERDICT_CODE[records[path]["clean_verdict"]] for path in clips], np.int8),
              "clean_top1": np.array([records[path]["clean_top1"] for path in clips], np.int32),
              "clip_no_snake": np.array([records[path]["no_snake_frames"] for path in clips], np.int32),
              "paths": np.array([path for path, _, _ in entries]), "schedules": np.array([name for _, name, _ in entries]),
              "offsets": np.concatenate([[0], np.cumsum(lengths)]).astype(np.int64),
              "processed": np.concatenate([frames["processed"] for _, _, frames in entries]).astype(np.int32),
              "shown": np.concatenate([frames["shown"] for _, _, frames in entries]).astype(np.float64),
              "snake": np.concatenate([frames["snake"] for _, _, frames in entries]).astype(bool)}
    for method in METHODS:
        arrays[f"verdict_{method}"] = np.array([VERDICT_CODE[label] for _, _, frames in entries
                                                for label in frames["verdicts"][method]], np.int8)
        arrays[f"top1_{method}"] = np.concatenate([frames["species"][method] for _, _, frames in entries]).astype(np.int32)
    np.savez_compressed(file, **arrays)


def _schedule_summary(clips, latency):
    first = clips[clips["method"] == METHODS[0]]
    effective_fps = float(first["effective_fps"].mean())
    return {"latency_seconds": float(np.mean(latency)), "latency_median_seconds": float(np.median(latency)),
            "effective_fps": effective_fps,
            "skipped_share": float(1 - (first["frames_processed"] / first["frames"]).mean()),
            "no_snake_processed_share": float((first["no_snake_processed"] / first["frames_processed"]).mean()),
            "meets_target_fps": effective_fps >= TARGET_FPS,
            "methods": {method: _method_summary(clips[clips["method"] == method]) for method in METHODS}}


def _no_snake_summary(records):
    missing = [record["no_snake_frames"] for record in records]
    total = sum(record["params"]["frames"] for record in records)
    return {"frames": int(sum(missing)), "frames_total": int(total), "share": sum(missing) / total if total else None,
            "clips_with_no_snake_frames": int(sum(count > 0 for count in missing)),
            "clean_no_snake": int(sum(record["clean_top1"] == NO_SPECIES for record in records))}


def _method_summary(clips):
    dangerous = clips[clips["dangerous"]]
    groups = photographer_groups(clips)
    return {**{kind: _label_summary(clips, kind, groups) for kind in ("verdict", "species")},
            "dangerous_clips": len(dangerous), "dangerous_ever_not_listed": _share(dangerous["ever_not_listed"]),
            "dangerous_final_not_listed": _share(dangerous["verdict_final_stable"] == NOT_LISTED)}


def _label_summary(clips, kind, groups):
    flips = clips[f"{kind}_flips"].to_numpy(dtype=float)
    stable_at = pd.to_numeric(clips[f"{kind}_time_to_stable"], errors="coerce").to_numpy(dtype=float)
    confirmed_at = pd.to_numeric(clips[f"{kind}_time_to_confirmed"], errors="coerce").to_numpy(dtype=float)
    correct = clips[f"{kind}_clip_correct"].to_numpy(dtype=bool)
    clean = clips[f"clean_{kind}_correct"].to_numpy(dtype=bool)
    stable = np.isfinite(stable_at)
    return {"mean_flips": float(flips.mean()), "median_flips": float(np.median(flips)),
            "o6_flips_share": _share(flips <= O6_MAX_FLIPS), "stable_share": _share(stable),
            "median_time_to_stable": float(np.median(stable_at[stable])) if stable.any() else None,
            "median_time_to_confirmed": float(np.median(confirmed_at[stable])) if stable.any() else None,
            "o6_stable_share": _share(stable & (np.nan_to_num(stable_at, nan=np.inf) <= O6_STABLE_WITHIN)),
            "o6_confirmed_share": _share(stable & (np.nan_to_num(confirmed_at, nan=np.inf) <= O6_STABLE_WITHIN)),
            "clip_correct": _share(correct),
            "clip_correct_ci": list(grouped_bootstrap(lambda rows: float(correct[rows].mean()), groups)),
            "frames_correct": float(clips[f"{kind}_frames_correct"].mean()),
            "clean_correct": _share(clean), "clip_correct_where_clean_correct": _share(correct[clean])}


def _share(flags):
    flags = np.asarray(flags, dtype=bool)
    return float(flags.mean()) if flags.size else None

In [8]:
OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """A kernel's output folder, or for "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


DATA = input_root("serpentguard-indian-snakes-clean")
RUN_OUTPUT = kernel_output(RUN_KERNEL)
runs = sorted(path.parent for path in RUN_OUTPUT.rglob(CHECKPOINT_FILE) if path.parent.name == RUN_ID)
assert len(runs) == 1, f"{RUN_KERNEL}'s output must hold one {RUN_ID} folder with {CHECKPOINT_FILE}; found {runs}"
RUN = runs[0]
# the leaves go over the front end's predicted masks, never SAM's
PREDS_OUTPUT = kernel_output(PREDS_KERNEL)
preds_files = sorted(PREDS_OUTPUT.rglob(FRONTEND_PREDS_FILE))
assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
PREDS = preds_files[0].parent
DECISION_INPUTS = (CALIBRATION_OUTPUTS, VAL_OUTPUTS, UNKNOWN_OUTPUTS, PROTOTYPES_FILE)
OUT = Path("/kaggle/working/video")

for path in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", RUN / CONFIG_FILE, RUN / HASH_FILE,
             *(RUN / name for name in DECISION_INPUTS)):
    assert path.exists(), f"missing input {path}"
cfg = load_config(RUN / CONFIG_FILE)
assert cfg.run_id == RUN_ID, f"{RUN} holds run {cfg.run_id}"
BOX_CROP_RUN = input_mode(cfg) == BOX_CROP  # F1: every frame goes through the D1 detector first
DETECTOR = None
if BOX_CROP_RUN:
    winners_files = sorted(PREDS_OUTPUT.rglob(WINNERS_FILE))
    assert len(winners_files) == 1, (f"{PREDS_KERNEL}'s output must hold notebook 13's frontends/{WINNERS_FILE}; "
                                     f"found {winners_files}")
    winners = json.loads(winners_files[0].read_text())
    DETECTOR = winners_files[0].parent / winners[D1]["weights"]
    assert DETECTOR.is_file(), f"the D1 detector {winners[D1]['weights']} is missing from {PREDS_KERNEL}'s output"
LAPTOP_SECONDS, LAPTOP_LATENCY = laptop_schedule(cfg, LATENCY_BACKEND)  # box_crop: X1's deployed pipeline
assert torch.cuda.is_available(), "no GPU in this session"
DEVICE = torch.device("cuda")
print(f"code {CODE}; run {RUN_ID} at {RUN}; predictions {PREDS}; detector {DETECTOR}; "
      f"{torch.cuda.get_device_name(DEVICE)}")
print(f"laptop step: {np.mean(LAPTOP_SECONDS) * 1000:.1f} ms per frame on average, from {LAPTOP_LATENCY}")

code /kaggle/working/repo/src; run H1_top1_s1 at /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-run-final/H1_top1_s1; predictions /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds; detector /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontends/D1/yolo26n-ft/detector.pt; Tesla T4
laptop step: 84.9 ms per frame on average, from {'file': 'results/m3/x1_laptop/package.csv', 'figure': 'pipeline_ms_1t', 'precision': 'deployment', 'encoder': 'mobilevit_s', 'snakeid_int8': 'fp16', 'size': 320, 'front_end': 'yolo26n-ft@320', 'front_end_precision': 'int8_weights', 'pipeline_ms_1t': 84.92068690247834, 'pipeline_ms_4t': 46.74908402375877, 'latency_device': 'laptop CPU (Apple M4 Pro)'}


In [9]:
# decision_report writes decision_metrics.json next to the outputs it reads, and the input mount is read-only
WORK_RUN = Path("/kaggle/working") / f"{cfg.run_id}_decision"
WORK_RUN.mkdir(parents=True, exist_ok=True)
for name in DECISION_INPUTS:
    shutil.copy2(RUN / name, WORK_RUN / name)
classes = pd.read_csv(DATA / "classes.csv")
metadata = pd.read_csv(DATA / "metadata.csv", dtype=ID_TYPES)
box_conf = None
if BOX_CROP_RUN:  # the front end's box confidences give the decision file its tau_snake
    preds = pd.read_csv(PREDS / FRONTEND_PREDS_FILE, usecols=["path", "box_conf"])
    box_conf = dict(zip(preds["path"], preds["box_conf"]))
decision = fitted_decision(WORK_RUN, classes, metadata, recall_bound=RECALL_BOUND, box_conf=box_conf)
TAU_SNAKE = json.loads((WORK_RUN / DECISION_FILE).read_text())["no_snake"].get("tau_snake")
assert TAU_SNAKE is not None or not BOX_CROP_RUN, "a box_crop run's decision file must hold tau_snake"
print(decision.thresholds, f"tau_snake {TAU_SNAKE}")

Thresholds(temperature=1.9086784115427977, temperature_who=0.9690096288687304, theta_d=0.05660591583161018, theta_who=inf, tau_unknown=0.40982445157291975, theta_conf=0.360998147721644, theta_v=0.5, theta_genus=0.5) tau_snake 0.102


In [10]:
frontend_preds = pd.read_csv(PREDS / FRONTEND_PREDS_FILE)
frame = with_files(build_frame(metadata, classes, ["val"], frontend_preds=frontend_preds, mask_source="predicted",
                               preds_root=PREDS), DATA)
photos = round_robin(frame, N_CLIPS, SEED)
photos = photos.head(LIMIT) if LIMIT else photos
dangerous = decision.tables.dangerous[photos["class_id"].to_numpy()]
print(f"{len(photos)} clips from {photos['species'].nunique()} classes, {int(dangerous.sum())} of them dangerous; "
      f"{photos['mask_valid'].mean():.0%} with a predicted mask for the leaves (the rest: the central ellipse)")

200 clips from 176 classes, 51 of them dangerous; 96% with a predicted mask for the leaves (the rest: the central ellipse)


In [11]:
started = time.time()
# Ultralytics runs the detector on the GPU when there is one
predict_frame = torch_predictor(RUN, DEVICE, detector=DETECTOR, tau_snake=TAU_SNAKE)
run_info = {"run_kernel": RUN_KERNEL, "run": cfg.run_id, "config": asdict(cfg), "recall_bound": RECALL_BOUND,
            "subset_hash": (RUN / HASH_FILE).read_text().strip(), "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(DEVICE), "preds_kernel": PREDS_KERNEL, "mask_source": "predicted",
            "detector": None if DETECTOR is None else str(DETECTOR.relative_to(PREDS_OUTPUT)),
            "tau_snake": TAU_SNAKE,
            "laptop_latency": LAPTOP_LATENCY}
schedules = {"every_frame": 0.0, "laptop_cpu": LAPTOP_SECONDS}
report = run_video(photos, predict_frame, decision, OUT, schedules, seed=SEED, run_info=run_info,
                   checkpoint=RUN / CHECKPOINT_FILE)
print(f"{report['clips']} of {report['photos']} clips in {(time.time() - started) / 60:.1f} min, "
      f"{len(report['errors'])} failed; leaves on {report['occluder_regions']}, {report['leaf_cover_below_range']} "
      f"clips under 30% cover; no snake on {report['no_snake']['frames']} of {report['no_snake']['frames_total']} "
      f"frames; wrote {OUT / VIDEO_FILE}, {OUT / CLIPS_FILE} and {OUT / FRAMES_FILE}")
for error in report["errors"]:
    print(error)

200 of 200 clips in 21.6 min, 0 failed; leaves on {'mask': 191, 'centre': 9}, 0 clips under 30% cover; no snake on 1898 of 30000 frames; wrote /kaggle/working/video/video_metrics.json, /kaggle/working/video/video_clips.csv and /kaggle/working/video/video_frames.npz


In [12]:
rows = []
for name, result in report["schedules"].items():
    for method, scores in result["methods"].items():
        verdict, species = scores["verdict"], scores["species"]
        rows.append({"schedule": name, "method": method, "effective FPS": result["effective_fps"],
                     "skipped": result["skipped_share"], "verdict flips (mean)": verdict["mean_flips"],
                     "verdict <= 1 flip": verdict["o6_flips_share"], "verdict stable <= 2 s": verdict["o6_stable_share"],
                     "verdict confirmed <= 2 s": verdict["o6_confirmed_share"],
                     "verdict clip correct": verdict["clip_correct"], "verdict clean correct": verdict["clean_correct"],
                     "verdict clip correct where clean correct": verdict["clip_correct_where_clean_correct"],
                     "species flips (mean)": species["mean_flips"], "species clip correct": species["clip_correct"],
                     "species clean correct": species["clean_correct"],
                     "dangerous ever 'not listed'": scores["dangerous_ever_not_listed"],
                     "dangerous final 'not listed'": scores["dangerous_final_not_listed"]})
summary = pd.DataFrame(rows).set_index(["schedule", "method"])
summary.to_csv(OUT / "summary.csv")
summary.round(3)

effective FPS  skipped  verdict flips (mean)  \
schedule    method                                                      
every_frame per_frame             15.0    0.000                33.230   
            ema                   15.0    0.000                15.485   
            vote_4_of_6           15.0    0.000                 7.665   
laptop_cpu  per_frame             11.8    0.213                26.195   
            ema                   11.8    0.213                12.065   
            vote_4_of_6           11.8    0.213                 6.085   

                         verdict <= 1 flip  verdict stable <= 2 s  \
schedule    method                                                  
every_frame per_frame                0.170                  0.595   
            ema                      0.330                  0.810   
            vote_4_of_6              0.385                  0.945   
laptop_cpu  per_frame                0.180                  0.630   
            ema                      0.370                  0.830   
            vote_4_of_6              0.405                  0.930   

                         verdict confirmed <= 2 s  verdict clip correct  \
schedule    method                                                        
every_frame per_frame                       0.555                 0.285   
            ema                             0.725                 0.305   
            vote_4_of_6                     0.870                 0.295   
laptop_cpu  per_frame                       0.555                 0.280   
            ema                             0.735                 0.290   
            vote_4_of_6                     0.870                 0.295   

                         verdict clean correct  \
schedule    method                               
every_frame per_frame                    0.585   
            ema                          0.585   
            vote_4_of_6                  0.585   
laptop_cpu  per_frame                    0.585   
            ema                          0.585   
            vote_4_of_6                  0.585   

                         verdict clip correct where clean correct  \
schedule    method                                                  
every_frame per_frame                                       0.453   
            ema                                             0.470   
            vote_4_of_6                                     0.462   
laptop_cpu  per_frame                                       0.444   
            ema                                             0.453   
            vote_4_of_6                                     0.462   

                         species flips (mean)  species clip correct  \
schedule    method                                                    
every_frame per_frame                  83.100                 0.070   
            ema                        33.390                 0.120   
            vote_4_of_6                16.015                 0.100   
laptop_cpu  per_frame                  65.440                 0.070   
            ema                        26.480                 0.125   
            vote_4_of_6                12.505                 0.095   

                         species clean correct  dangerous ever 'not listed'  \
schedule    method                                                            
every_frame per_frame                    0.365                        0.294   
            ema                          0.365                        0.078   
            vote_4_of_6                  0.365                        0.078   
laptop_cpu  per_frame                    0.365                        0.294   
            ema                          0.365                        0.078   
            vote_4_of_6                  0.365                        0.059   

                         dangerous final 'not listed'  
schedule    method                                     
every_frame per_fr